In [4]:
import pandas as pd
import numpy as np
from sklearn.metrics import f1_score
from evaluate import load


def parse_file_info(file_path):
    """Estrae prompting method e input type dal nome del file."""
    if "zero_shot" in file_path:
        prompting_method = "Zero-Shot"
    elif "chain_of_thought" in file_path:
        prompting_method = "Chain-of-Thought"
    elif "fine_tuning" in file_path:
        prompting_method = "Fine-Tuning"
    else:
        prompting_method = "Unknown"

    if "_context_reply.csv" in file_path:
        input_type = "context_reply"
    elif "_last_u1_message.csv" in file_path:
        input_type = "context"
    elif "_conversation.csv" in file_path:
        input_type = "conversation"
    else:
        input_type = "unknown"

    return input_type, prompting_method


def compute_metrics(df, accuracy_metric, precision_metric, recall_metric, f1_metric):
    """Calcola metriche standard su un file di predizioni."""
    df = df.copy()
    df["model_answer"] = df["model_answer"].astype(int)
    df["label"] = df["label"].astype(int)

    accuracy = accuracy_metric.compute(predictions=df["model_answer"], references=df["label"])["accuracy"]

    precision = precision_metric.compute(
        predictions=df["model_answer"],
        references=df["label"],
        pos_label=1,
        average="binary"
    )["precision"]

    recall = recall_metric.compute(
        predictions=df["model_answer"],
        references=df["label"],
        pos_label=1,
        average="binary"
    )["recall"]

    f1 = f1_metric.compute(
        predictions=df["model_answer"],
        references=df["label"],
        pos_label=1,
        average="binary"
    )["f1"]

    total = len(df)
    correct = int((df["model_answer"] == df["label"]).sum())
    label_0 = int((df["label"] == 0).sum())
    label_1 = int((df["label"] == 1).sum())
    pred_0 = int((df["model_answer"] == 0).sum())
    pred_1 = int((df["model_answer"] == 1).sum())

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "total": total,
        "correct": correct,
        "label_0": label_0,
        "label_1": label_1,
        "pred_0": pred_0,
        "pred_1": pred_1,
    }


def print_console_results(results_data):
    """Stampa i risultati aggregati (metriche) in console."""
    for dataset_name, dataset_results in results_data.items():
        print("\n" + "=" * 100)
        print(f"DATASET: {dataset_name}".center(100))
        print("=" * 100)

        for model_name, model_results in dataset_results.items():
            print(f"\n{'MODEL: ' + model_name}".center(100, '-'))
            print(f"\n{'Technique':<20} {'Input Type':<16} {'Acc':<8} {'Prec':<8} {'Rec':<8} {'F1':<8} {'Samples':<10}")
            print("-" * 100)

            for r in model_results:
                m = r["metrics"]
                print(
                    f"{r['technique']:<20} {r['input_type']:<16} "
                    f"{m['accuracy']:<8.4f} {m['precision']:<8.4f} {m['recall']:<8.4f} {m['f1']:<8.4f} "
                    f"{m['total']:<10}"
                )


def bootstrap_f1_test(df_context, df_conversation, n_bootstrap=10000, alpha=0.05, seed=42):
    """
    Paired bootstrap su ΔF1 = F1(context) - F1(conversation).
    Richiede: stessi esempi nello stesso ordine (stesse label riga-per-riga).
    """
    df_context = df_context.copy()
    df_conversation = df_conversation.copy()

    df_context["label"] = df_context["label"].astype(int)
    df_conversation["label"] = df_conversation["label"].astype(int)
    df_context["model_answer"] = df_context["model_answer"].astype(int)
    df_conversation["model_answer"] = df_conversation["model_answer"].astype(int)

    if len(df_context) != len(df_conversation):
        raise ValueError("Context e conversation devono avere lo stesso numero di righe.")

    if not (df_context["label"].values == df_conversation["label"].values).all():
        raise ValueError(
            "Le label non coincidono riga-per-riga tra context e conversation. "
            "Assicurati che siano lo stesso test set nello stesso ordine."
        )

    y_true = df_context["label"].values
    y_pred_ctx = df_context["model_answer"].values
    y_pred_conv = df_conversation["model_answer"].values

    n = len(y_true)
    rng = np.random.default_rng(seed)
    deltas = np.empty(n_bootstrap, dtype=float)

    for b in range(n_bootstrap):
        idx = rng.integers(0, n, n)
        f1_ctx = f1_score(y_true[idx], y_pred_ctx[idx], average="binary", pos_label=1)
        f1_conv = f1_score(y_true[idx], y_pred_conv[idx], average="binary", pos_label=1)
        deltas[b] = f1_ctx - f1_conv

    ci_low = float(np.percentile(deltas, 100 * alpha / 2))
    ci_high = float(np.percentile(deltas, 100 * (1 - alpha / 2)))
    return {
        "delta_mean": float(deltas.mean()),
        "ci_low": ci_low,
        "ci_high": ci_high,
        "significant": bool(not (ci_low <= 0.0 <= ci_high))
    }


def build_pairs_for_significance(file_paths):
    """
    Costruisce le coppie (technique -> (context_path, conversation_path))
    Ignora *_context_reply.csv.
    """
    pairs = {}

    for fp in file_paths:
        input_type, technique = parse_file_info(fp)

        if input_type == "context_reply":
            continue  # non lo usiamo nel confronto context vs conversation

        if technique not in pairs:
            pairs[technique] = {"context": None, "conversation": None}

        if input_type == "context":
            pairs[technique]["context"] = fp
        elif input_type == "conversation":
            pairs[technique]["conversation"] = fp

    # tieni solo technique complete (context+conversation)
    out = {}
    for technique, d in pairs.items():
        if d["context"] is not None and d["conversation"] is not None:
            out[technique] = (d["context"], d["conversation"])
    return out


def main():
    # Metriche HF Evaluate
    print("Loading evaluation metrics...")
    accuracy_metric = load("accuracy")
    precision_metric = load("precision")
    recall_metric = load("recall")
    f1_metric = load("f1")

    # I TUOI FILE (come da messaggio)
    files = {
        'CSC': {
            "EuroBERT_EuroBERT-210m": [
                "./output/encoders/EuroBERT_EuroBERT-210m/CSC/last_u1_message/predictions/fine_tuning_EuroBERT_EuroBERT-210m_CSC_last_u1_message.csv",
                "./output/encoders/EuroBERT_EuroBERT-210m/CSC/conversation/predictions/fine_tuning_EuroBERT_EuroBERT-210m_CSC_conversation.csv",
            ],
            "FacebookAI_xlm-roberta-large": [
                "./output/encoders/FacebookAI_xlm-roberta-large/CSC/last_u1_message/predictions/fine_tuning_FacebookAI_xlm-roberta-large_CSC_last_u1_message.csv",
                "./output/encoders/FacebookAI_xlm-roberta-large/CSC/conversation/predictions/fine_tuning_FacebookAI_xlm-roberta-large_CSC_conversation.csv",
            ],
            "Qwen_Qwen2.5-7B-Instruct": [
                "./output/decoders/Qwen_Qwen2.5-7B-Instruct/zero_shot_Qwen_Qwen2.5-7B-Instruct_CSC_last_u1_message.csv",
                "./output/decoders/Qwen_Qwen2.5-7B-Instruct/zero_shot_Qwen_Qwen2.5-7B-Instruct_CSC_conversation.csv",
                "./output/decoders/Qwen_Qwen2.5-7B-Instruct/chain_of_thought_Qwen_Qwen2.5-7B-Instruct_CSC_last_u1_message.csv",
                "./output/decoders/Qwen_Qwen2.5-7B-Instruct/chain_of_thought_Qwen_Qwen2.5-7B-Instruct_CSC_conversation.csv",
            ],
            "mistralai_Ministral-8B-Instruct-2410": [
                "./output/decoders/mistralai_Ministral-8B-Instruct-2410/zero_shot_mistralai_Ministral-8B-Instruct-2410_CSC_last_u1_message.csv",
                "./output/decoders/mistralai_Ministral-8B-Instruct-2410/zero_shot_mistralai_Ministral-8B-Instruct-2410_CSC_conversation.csv",
                "./output/decoders/mistralai_Ministral-8B-Instruct-2410/chain_of_thought_mistralai_Ministral-8B-Instruct-2410_CSC_last_u1_message.csv",
                "./output/decoders/mistralai_Ministral-8B-Instruct-2410/chain_of_thought_mistralai_Ministral-8B-Instruct-2410_CSC_conversation.csv",
            ],
              "Gemma-2-9b-it": [
                "./output/decoders/google_gemma-2-9b-it/zero_shot_google_gemma-2-9b-it_CSC_last_u1_message.csv",
                "./output/decoders/google_gemma-2-9b-it/zero_shot_google_gemma-2-9b-it_CSC_conversation.csv",
                "./output/decoders/google_gemma-2-9b-it/chain_of_thought_google_gemma-2-9b-it_CSC_last_u1_message.csv",
                "./output/decoders/google_gemma-2-9b-it/chain_of_thought_google_gemma-2-9b-it_CSC_conversation.csv",
            ],
        },
        'MultiPICo': {
            "FacebookAI_xlm-roberta-large": [
                "./output/encoders/FacebookAI_xlm-roberta-large/MultiPICo/last_u1_message/predictions/fine_tuning_FacebookAI_xlm-roberta-large_MultiPICo_last_u1_message.csv",
                "./output/encoders/FacebookAI_xlm-roberta-large/MultiPICo/conversation/predictions/fine_tuning_FacebookAI_xlm-roberta-large_MultiPICo_conversation.csv",
            ],
            "EuroBERT_EuroBERT-210m": [
                "./output/encoders/EuroBERT_EuroBERT-210m/MultiPICo/last_u1_message/predictions/fine_tuning_EuroBERT_EuroBERT-210m_MultiPICo_last_u1_message.csv",
                "./output/encoders/EuroBERT_EuroBERT-210m/MultiPICo/conversation/predictions/fine_tuning_EuroBERT_EuroBERT-210m_MultiPICo_conversation.csv",
            ],
            "Qwen_Qwen2.5-7B-Instruct": [
                "./output/decoders/Qwen_Qwen2.5-7B-Instruct/zero_shot_Qwen_Qwen2.5-7B-Instruct_MultiPICo_last_u1_message.csv",
                "./output/decoders/Qwen_Qwen2.5-7B-Instruct/zero_shot_Qwen_Qwen2.5-7B-Instruct_MultiPICo_conversation.csv",
                "./output/decoders/Qwen_Qwen2.5-7B-Instruct/chain_of_thought_Qwen_Qwen2.5-7B-Instruct_MultiPICo_last_u1_message.csv",
                "./output/decoders/Qwen_Qwen2.5-7B-Instruct/chain_of_thought_Qwen_Qwen2.5-7B-Instruct_MultiPICo_conversation.csv",
            ],
            "mistralai_Ministral-8B-Instruct-2410": [
                "./output/decoders/mistralai_Ministral-8B-Instruct-2410/zero_shot_mistralai_Ministral-8B-Instruct-2410_MultiPICo_last_u1_message.csv",
                "./output/decoders/mistralai_Ministral-8B-Instruct-2410/zero_shot_mistralai_Ministral-8B-Instruct-2410_MultiPICo_conversation.csv",
                "./output/decoders/mistralai_Ministral-8B-Instruct-2410/chain_of_thought_mistralai_Ministral-8B-Instruct-2410_MultiPICo_last_u1_message.csv",
                "./output/decoders/mistralai_Ministral-8B-Instruct-2410/chain_of_thought_mistralai_Ministral-8B-Instruct-2410_MultiPICo_conversation.csv",
            ],
            "Gemma-2-9b-it": [
                "./output/decoders/google_gemma-2-9b-it/zero_shot_google_gemma-2-9b-it_MultiPICo_last_u1_message.csv",
                "./output/decoders/google_gemma-2-9b-it/zero_shot_google_gemma-2-9b-it_MultiPICo_conversation.csv",
                "./output/decoders/google_gemma-2-9b-it/chain_of_thought_google_gemma-2-9b-it_MultiPICo_last_u1_message.csv",
                "./output/decoders/google_gemma-2-9b-it/chain_of_thought_google_gemma-2-9b-it_MultiPICo_conversation.csv",
            ],
        }
    }

    # 1) Calcolo metriche per tutti i file
    results_data = {}
    print("\nEvaluating models...\n")

    for dataset_name, dataset_models in files.items():
        results_data[dataset_name] = {}

        for model_name, file_paths in dataset_models.items():
            results_data[dataset_name][model_name] = []

            for fp in file_paths:
                input_type, technique = parse_file_info(fp)

                df = pd.read_csv(fp)
                metrics = compute_metrics(df, accuracy_metric, precision_metric, recall_metric, f1_metric)

                results_data[dataset_name][model_name].append({
                    "file_path": fp,
                    "input_type": input_type,
                    "technique": technique,
                    "metrics": metrics
                })

                print(f"✓ Evaluated: {dataset_name} - {model_name} - {technique} - {input_type}")

    print_console_results(results_data)

    # 2) Test di significatività F1: context vs conversation (sullo stesso dataset)
    print("\n" + "=" * 100)
    print("SIGNIFICANCE TEST (Paired Bootstrap) - F1: context vs conversation".center(100))
    print("=" * 100)

    for dataset_name, dataset_models in files.items():
        print("\n" + "-" * 100)
        print(f"DATASET: {dataset_name}".center(100))
        print("-" * 100)

        for model_name, file_paths in dataset_models.items():
            pairs = build_pairs_for_significance(file_paths)

            if not pairs:
                print(f"\nMODEL: {model_name}")
                print("  (Nessuna coppia completa context+conversation trovata per fare il test.)")
                continue

            print(f"\nMODEL: {model_name}")
            for technique, (ctx_path, conv_path) in pairs.items():
                df_ctx = pd.read_csv(ctx_path)
                df_conv = pd.read_csv(conv_path)

                # F1 puntuali sull'intero test set
                f1_ctx = f1_score(
                    df_ctx["label"].astype(int),
                    df_ctx["model_answer"].astype(int),
                    average="binary",
                    pos_label=1
                )
                f1_conv = f1_score(
                    df_conv["label"].astype(int),
                    df_conv["model_answer"].astype(int),
                    average="binary",
                    pos_label=1
                )

                test = bootstrap_f1_test(df_ctx, df_conv, n_bootstrap=10000, alpha=0.05, seed=42)

                print(f"  Technique: {technique}")
                print(f"    F1(context)      = {f1_ctx:.4f}")
                print(f"    F1(conversation) = {f1_conv:.4f}")
                print(f"    ΔF1 (ctx-conv)   = {test['delta_mean']:.4f}")
                print(f"    95% CI ΔF1       = [{test['ci_low']:.4f}, {test['ci_high']:.4f}]")
                print(f"    Significativo?   = {'YES' if test['significant'] else 'NO'}")

    print("\n" + "=" * 100)
    print("✓ Report generated successfully!")
    print("=" * 100)


if __name__ == "__main__":
    main()


Loading evaluation metrics...

Evaluating models...

✓ Evaluated: CSC - EuroBERT_EuroBERT-210m - Fine-Tuning - context
✓ Evaluated: CSC - EuroBERT_EuroBERT-210m - Fine-Tuning - conversation
✓ Evaluated: CSC - FacebookAI_xlm-roberta-large - Fine-Tuning - context
✓ Evaluated: CSC - FacebookAI_xlm-roberta-large - Fine-Tuning - conversation
✓ Evaluated: CSC - Qwen_Qwen2.5-7B-Instruct - Zero-Shot - context
✓ Evaluated: CSC - Qwen_Qwen2.5-7B-Instruct - Zero-Shot - conversation
✓ Evaluated: CSC - Qwen_Qwen2.5-7B-Instruct - Chain-of-Thought - context
✓ Evaluated: CSC - Qwen_Qwen2.5-7B-Instruct - Chain-of-Thought - conversation
✓ Evaluated: CSC - mistralai_Ministral-8B-Instruct-2410 - Zero-Shot - context
✓ Evaluated: CSC - mistralai_Ministral-8B-Instruct-2410 - Zero-Shot - conversation
✓ Evaluated: CSC - mistralai_Ministral-8B-Instruct-2410 - Chain-of-Thought - context
✓ Evaluated: CSC - mistralai_Ministral-8B-Instruct-2410 - Chain-of-Thought - conversation
✓ Evaluated: CSC - Gemma-2-9b-it - Ze